In [12]:
from pytrends.request import TrendReq
import pandas as pd
from datetime import datetime, timedelta
import time
import requests
from vmdpy import VMD
from pandas_datareader import data as pdr
import talib
import warnings
import numpy as np
warnings.filterwarnings('ignore')


VMD

Цены

In [ ]:
API_KEY = ##
BASE_URL = "https://min-api.cryptocompare.com/data/v2/"
def get_crypto_data_paged(fsym="BTC", tsym="USD", interval="histoday", start_date_str='2020-01-01'):
    all_data = []
    toTs = int(time.time())  # начинаем с "сейчас"
    limit = 2000
    start_ts_limit = int(datetime.strptime(start_date_str, "%Y-%m-%d").timestamp())  # целевая нижняя граница

    while True:
        url = f"{BASE_URL}{interval}"
        params = {
            "fsym": fsym,
            "tsym": tsym,
            "limit": limit,
            "toTs": toTs,
            "api_key": API_KEY
        }

        response = requests.get(url, params=params)
        data = response.json()

        if data["Response"] != "Success":
            print("Ошибка:", data.get("Message", "Unknown error"))
            break

        chunk = data["Data"]["Data"]
        if not chunk:
            break

        all_data.extend(chunk)

        # Обновляем toTs на самый ранний момент в текущем чанке
        min_ts = min(item["time"] for item in chunk)
        if min_ts <= start_ts_limit:
            break
        toTs = min_ts - 1  # чтобы избежать дубликата

        print(f"Загружено {len(all_data)} точек, до {datetime.utcfromtimestamp(min_ts).strftime('%Y-%m-%d')}")

        time.sleep(1)  # Чтобы не попасть под лимит API

    # Преобразуем в DataFrame
    btc_price_2020 = pd.DataFrame(all_data).drop_duplicates(subset="time")
    btc_price_2020["time"] = pd.to_datetime(btc_price_2020["time"], unit="s")
    return btc_price_2020.sort_values("time").reset_index(drop=True)    

# Пример: 5 лет дневных данных (365 * 5 = 1825)
BTC_PRICE = get_crypto_data_paged(fsym="BTC",interval="histoday")
SOL_PRICE=get_crypto_data_paged(fsym="SOL",interval="histoday")

# Пример: 2000 часов


Загружено 2001 точек, до 2020-08-23
Загружено 2001 точек, до 2020-08-23


In [18]:
BTC_PRICE.to_csv(r'C:\Users\User\my\py\practise\solana\data\btc_price_from_2020.csv',index=False)
SOL_PRICE.to_csv(r'C:\Users\User\my\py\practise\solana\data\sol_price_from_2020.csv',index=False)

In [2]:
SOL_PRICE=pd.read_csv(r'C:\Users\User\my\py\practise\solana\data\sol_price_from_2020.csv')

In [3]:
SOL_PRICE

,time,high,low,open,volumefrom,volumeto,close,conversionType,conversionSymbol
0,2015-03-02,0.00,0.00,0.00,0.00,0.000000e+00,0.00,direct,NaN
1,2015-03-03,0.00,0.00,0.00,0.00,0.000000e+00,0.00,direct,NaN
2,2015-03-04,0.00,0.00,0.00,0.00,0.000000e+00,0.00,direct,NaN
3,2015-03-05,0.00,0.00,0.00,0.00,0.000000e+00,0.00,direct,NaN
4,2015-03-06,0.00,0.00,0.00,0.00,0.000000e+00,0.00,direct,NaN
...,...,...,...,...,...,...,...,...,...
3997,2026-02-09,88.63,82.81,86.94,3180782.56,2.736487e+08,86.72,direct,NaN
3998,2026-02-10,87.44,81.88,86.72,2565542.88,2.160229e+08,82.94,direct,NaN
3999,2026-02-11,84.34,78.01,82.94,3889278.38,3.138150e+08,79.24,direct,NaN
4000,2026-02-12,82.19,76.54,79.24,3420507.03,2.714720e+08,78.33,direct,NaN


Market regimes

In [23]:
df_regimes=SOL_PRICE.copy()
df_regimes['price']=df_regimes['volumeto']/df_regimes['volumefrom']
df_regimes.rename(columns={"time": "date"}, inplace=True)
df_regimes=df_regimes[['date','close']]
df_regimes=df_regimes.dropna()
df_regimes["ret_1"] = np.log(df_regimes["close"] / df_regimes["close"].shift(1))
df_regimes["trend_30"] = df_regimes["close"] / df_regimes["close"].shift(30) - 1
df_regimes["ema_30"] = df_regimes["close"].ewm(span=30, adjust=False).mean()
df_regimes["trend_ema"] = df_regimes["close"] / df_regimes["ema_30"] - 1
df_regimes["vol_20"] = df_regimes["ret_1"].rolling(20).std()
df_regimes=df_regimes.dropna()

In [25]:

df_regimes=df_regimes[df_regimes['date']>'2021-01-01']
df_regimes=df_regimes[df_regimes['date']<'2025-11-01']
df_regimes

,date,close,ret_1,trend_30,ema_30,trend_ema,vol_20
2133,2021-01-02,1.799,-0.026332,-0.142517,1.642396,0.095351,0.094607
2134,2021-01-03,2.185,0.194385,0.181720,1.677403,0.302609,0.103497
2135,2021-01-04,2.491,0.131068,0.258081,1.729893,0.439973,0.104975
2136,2021-01-05,2.158,-0.143502,0.164598,1.757513,0.227872,0.110977
2137,2021-01-06,1.925,-0.114256,0.052488,1.768318,0.088605,0.113383
...,...,...,...,...,...,...,...
3892,2025-10-27,198.710,-0.006821,-0.023922,200.459631,-0.008728,0.054919
3893,2025-10-28,194.230,-0.022803,-0.079130,200.057719,-0.029130,0.053983
3894,2025-10-29,193.980,-0.001288,-0.088911,199.665608,-0.028476,0.053603
3895,2025-10-30,184.690,-0.049076,-0.115257,198.699440,-0.070506,0.041772


In [25]:
def build_rolling_vmd_features(
    df: pd.DataFrame,
    price_col: str = "close",
    date_col: str = "date",
    window: int = 200,
    alpha: float = 2000,
    tau: float = 0,
    K: int = 10,
    DC: int = 0,
    init: int = 1,
    tol: float = 1e-7,
    use_current_bar: bool = True,
    verbose: bool = True,
) -> pd.DataFrame:
    df = df.copy()

    if date_col in df.columns:
        df[date_col] = pd.to_datetime(df[date_col]).dt.normalize()
        df = df.sort_values(date_col).reset_index(drop=True)

    prices = df[price_col].astype(float).values
    rows = []

    start_idx = window - 1 if use_current_bar else window

    for t in range(start_idx, len(df)):
        if use_current_bar:
            signal = prices[t - window + 1 : t + 1]
        else:
            signal = prices[t - window : t]

        if len(signal) != window:
            continue

        try:
            u, u_hat, omega = VMD(signal, alpha, tau, K, DC, init, tol)

            row = {date_col: df.iloc[t][date_col]}
            for i in range(K):
                row[f"vmd_{i+1}"] = u[i, -1]

            rows.append(row)

        except Exception as e:
            if verbose:
                print(f"VMD error at index {t}: {e}")

        if verbose and (t % 100 == 0):
            print(f"Processed {t}/{len(df)-1}")

    df_vmd_roll = pd.DataFrame(rows)
    return df_vmd_roll

In [21]:
df_base = pd.read_csv(r'C:\Users\User\my\py\practise\solana\data\sol_price_from_2020.csv')
df_base=df_base[df_base['time']>'2020-04-10']
df_base['time'] = pd.to_datetime(df_base['time'])
df_base.rename(columns={"time": "date"}, inplace=True)

price = df_base['close'].values


In [26]:
df_vmd_roll = build_rolling_vmd_features(
    df=df_base,
    price_col="close",
    date_col="date",
    window=200,
    alpha=2000,
    tau=0,
    K=10,
    DC=0,
    init=1,
    tol=1e-7,
    use_current_bar=True,
    verbose=True
)
df_vmd_roll.to_csv(r'C:\Users\User\my\py\practise\solana\data\SOL_VMD.csv',index=False)

Processed 200/2134
Processed 300/2134
Processed 400/2134
Processed 500/2134
Processed 600/2134
Processed 700/2134
Processed 800/2134
Processed 900/2134
Processed 1000/2134
Processed 1100/2134
Processed 1200/2134
Processed 1300/2134
Processed 1400/2134
Processed 1500/2134
Processed 1600/2134
Processed 1700/2134
Processed 1800/2134
Processed 1900/2134
Processed 2000/2134
Processed 2100/2134


Простые дейли фичи по биржевым потокам

In [32]:
df_final=pd.read_csv(r"C:\Users\User\my\py\practise\solana\combined_from_01_2020.csv")
df_final= df_final.drop_duplicates().reset_index(drop=True)
df_final["Human Time"] = pd.to_datetime(df_final["Human Time"], utc=True, errors="coerce")
df_final=df_final.sort_values(by='Human Time').reset_index(drop=True)
df_final["date"] = df_final["Human Time"].dt.date
df_final['Amount']=df_final['Amount']/1000000000
df_final = df_final[
    ['Human Time','date','Amount', 'Flow', 'Value', 'From', 'To']
]
df_final=df_final.dropna()

In [34]:
df_final["signed_amount"] = df_final["Amount"].where(
    df_final["Flow"] == "in",
    -df_final["Amount"]
)
daily = df_final.groupby("date").agg(
    total_amount=("Amount", "sum"),
    net_flow=("signed_amount", "sum")
).reset_index()

daily["flow_ratio"] = daily["net_flow"] / daily["total_amount"]

In [41]:
daily.to_csv(r'C:\Users\User\my\py\practise\solana\data\flows_from2020.csv',index=False)

S&P500

In [27]:
df_sp500=pd.read_csv(r"C:\Users\User\my\py\practise\solana\data\sp500_2020.csv")
df_sp500['Дата']=pd.to_datetime(df_sp500['Дата'])
df_sp500.rename(columns={"Дата": "date"}, inplace=True)
df_sp500.rename(columns={"Цена": "sp500_close"}, inplace=True)
df_sp500=df_sp500[['date','sp500_close']]
df_sp500['sp500_close'] = (
    df_sp500['sp500_close']
    .str.replace('.', '', regex=False)   # убрать разделитель тысяч
    .str.replace(',', '.', regex=False)  # заменить десятичный разделитель
    .astype(float)                       # преобразовать в float
)
df_sp500 = df_sp500.sort_values('date').reset_index(drop=True)

# полный ежедневный календарь
full_dates = pd.date_range(df_sp500['date'].min(), df_sp500['date'].max(), freq='D')
df_sp500 = (
    df_sp500
    .set_index('date')
    .reindex(full_dates)
    .rename_axis('date')
    .reset_index()
)

df_sp500['sp500_close'] = df_sp500['sp500_close'].ffill()
df_sp500.to_csv(r"C:\Users\User\my\py\practise\solana\data\sp500_2020_preprocessed.csv",index=False)

Ставка ФРС США

In [12]:
end_date = datetime.today()
start_date = end_date - timedelta(days=365*7)

# загрузка ставки ФРС (Fed Funds Rate)
fed_rate_daily = pdr.DataReader(
    'DFF',    # daily federal funds rate
    'fred',
    start=start_date,
    end=end_date
)

fed_rate_daily = fed_rate_daily.rename(columns={'DFF': 'fed_rate'})
fed_rate_daily = fed_rate_daily.reset_index()
fed_rate_daily = fed_rate_daily.rename(columns={
    'DATE': 'date',
    'FEDFUNDS': 'fed_rate'
})
fed_rate_daily.to_csv(r'C:\Users\User\my\py\practise\solana\data\fed_rate_daily_2020.csv',index=False)

US dollar index

In [30]:
start = datetime.today() - timedelta(days=365*7)
end = datetime.today()

dxy = (
    pdr.DataReader('DTWEXBGS', 'fred', start, end)
    .reset_index()
    .rename(columns={
        'DATE': 'date',
        'DTWEXBGS': 'dxy'
    })
)
dxy = dxy.sort_values('date').reset_index(drop=True)

# полный ежедневный календарь
full_dates = pd.date_range(dxy['date'].min(), dxy['date'].max(), freq='D')
dxy = (
    dxy
    .set_index('date')
    .reindex(full_dates)
    .rename_axis('date')
    .reset_index()
)

dxy['dxy'] = dxy['dxy'].ffill()
dxy.to_csv(r'C:\Users\User\my\py\practise\solana\data\dollar_index_2020.csv',index=False)


Technical analysis

In [20]:

def generate_technical(df):
    open_ = df['open']
    high = df['high']
    low = df['low']
    close = df['close']


    df['CDL_ENGULFING'] = talib.CDLENGULFING(open_, high, low, close)              # Поглощение
    df['CDL_HAMMER'] = talib.CDLHAMMER(open_, high, low, close)                    # Молот
    df['CDL_HANGINGMAN'] = talib.CDLHANGINGMAN(open_, high, low, close)            # Повешенный
    df['CDL_DOJI'] = talib.CDLDOJI(open_, high, low, close)                        # Дожи
    df['CDL_MORNINGSTAR'] = talib.CDLMORNINGSTAR(open_, high, low, close)          # Утренняя звезда
    df['CDL_EVENINGSTAR'] = talib.CDLEVENINGSTAR(open_, high, low, close)          # Вечерняя звезда
    df['CDL_3WHITESOLDIERS'] = talib.CDL3WHITESOLDIERS(open_, high, low, close)    # Три белых солдата
    df['CDL_3BLACKCROWS'] = talib.CDL3BLACKCROWS(open_, high, low, close)          # Три чёрные вороны
    df['CDL_PIERCING'] = talib.CDLPIERCING(open_, high, low, close)                # Линия прорыва
    df['CDL_DARKCLOUD'] = talib.CDLDARKCLOUDCOVER(open_, high, low, close)         # Тёмное облако


    # EMA
    df['vwap']=df['volumefrom']/df['volumeto']
    df['EMA_5'] = df['vwap'].ewm(span=5, adjust=False).mean()
    df['EMA_30'] = df['vwap'].ewm(span=30, adjust=False).mean()
    df['EMA_100'] = df['vwap'].ewm(span=100, adjust=False).mean()
    ema12 = df['vwap'].ewm(span=12, adjust=False).mean()
    ema26 = df['vwap'].ewm(span=26, adjust=False).mean()
    df['macd'] = ema12 - ema26
    df['macd_signal'] = df['macd'].ewm(span=9).mean()
    df['macd_diff'] = df['macd'] - df['macd_signal']
    window = 30  
    eps = 1e-9
    #macd,macd_z
    rolling_mean = df['macd'].rolling(window).mean()
    rolling_std = df['macd'].rolling(window).std()
    df['macd_z'] = (df['macd'] - rolling_mean) / (rolling_std + eps)

    delta = df['vwap'].diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    #дивергенция
    price_slope = (df['vwap'] - df['vwap'].shift(5)) / 5
    macd_slope = (df['macd'] - df['macd'].shift(5)) / 5

    # Фича: расхождение направлений
    df['macd_price_slope_diff'] = price_slope - macd_slope
    df['divergence_strength'] = df['macd_price_slope_diff'].rolling(window=3).sum()

    #RSI
    vwap_gain = gain.rolling(window=14).mean()
    vwap_loss = loss.rolling(window=14).mean()
    rs = vwap_gain / vwap_loss
    df['rsi'] = 100 - (100 / (1 + rs))

#Bolinger Bands
    window = 20
    df['bb_ma'] = df['vwap'].rolling(window).mean()
    df['bb_std'] = df['vwap'].rolling(window).std()
    df['bb_upper'] = df['bb_ma'] + 2 * df['bb_std']
    df['bb_lower'] = df['bb_ma'] - 2 * df['bb_std']
    #рассчет расстояния до нижней линии; рассчет расстояния до верхней линиии
    df['bb_dist_upper'] = (df['bb_upper'] - df['vwap'])/df['vwap']
    df['bb_dist_lower'] = (df['vwap'] - df['bb_lower'])/df['vwap']
    df['bb_pos'] = (df['vwap'] - df['bb_lower']) / (df['bb_upper'] - df['bb_lower'])
    df['bb_break_upper'] = df.apply(lambda row: 1 if row['vwap'] > row['bb_upper'] else 0, axis=1)
    df['bb_break_lower'] = df.apply(lambda row: 1 if row['vwap'] < row['bb_lower'] else 0, axis=1)

    accumulated = []
    acc = 0
    #накопление для линий болинджера
    for val in df['bb_break_lower']:
        if val == 1:
            acc += 1
            accumulated.append(acc)
        elif val!=1 and acc!=0:
            acc -=1
            accumulated.append(acc)
        else:
            acc = 0
            accumulated.append(acc)

    df['bb_break_lower_cumulative'] = accumulated

    accumulated1 = []
    acc1 = 0

    for val in df['bb_break_upper']:
        if val == 1:
            acc1 += 1
            accumulated1.append(acc1)
        elif val!=1 and acc1!=0:
            acc1 -=1
            accumulated1.append(acc1)
        else:
            acc1=0
            accumulated1.append(acc1)

    df['bb_break_upper_cumulative'] = accumulated1

    # df=df.sort_values(by='time')
    # === OBV (On-Balance volumefrom) ===
    obv = [0]
    for i in range(1, len(df)):
        if df.loc[i, 'close'] > df.loc[i - 1, 'close']:
            obv.append(obv[-1] + df.loc[i, 'volumefrom'])
        elif df.loc[i, 'close'] < df.loc[i - 1, 'close']:
            obv.append(obv[-1] - df.loc[i, 'volumefrom'])
        else:
            obv.append(obv[-1])
    df['OBV'] = obv

    # === Accumulation/Distribution Line (A/D) ===
    mf_multiplier = ((df['close'] - df['low']) - (df['high'] - df['close'])) / (df['high'] - df['low'] + 1e-9)
    mf_volumefrom = mf_multiplier * df['volumefrom']
    df['AD'] = mf_volumefrom.cumsum()

    # === Chaikin Money Flow (CMF) ===
    period_cmf = 20
    mfv_cmf = mf_multiplier * df['volumefrom']
    cmf = mfv_cmf.rolling(period_cmf).sum() / df['volumefrom'].rolling(period_cmf).sum()
    df['CMF'] = cmf

    # === Money Flow Index (MFI) ===
    period_mfi = 14
    typical_price = (df['high'] + df['low'] + df['close']) / 3
    money_flow = typical_price * df['volumefrom']
    pos_flow = []
    neg_flow = []
    for i in range(1, len(typical_price)):
        if typical_price[i] > typical_price[i - 1]:
            pos_flow.append(money_flow[i])
            neg_flow.append(0)
        elif typical_price[i] < typical_price[i - 1]:
            pos_flow.append(0)
            neg_flow.append(money_flow[i])
        else:
            pos_flow.append(0)
            neg_flow.append(0)

    pos_flow = pd.Series(pos_flow).rolling(period_mfi).sum()
    neg_flow = pd.Series(neg_flow).rolling(period_mfi).sum()
    mfi = 100 - (100 / (1 + (pos_flow / (neg_flow + 1e-9))))
    df['MFI'] = mfi.reindex(df.index[1:], method='bfill')  # сдвиг из-за i=1

    df['volume_avg'] = df['volumefrom'].rolling(window=20).mean()
    # === Force Index ===
    period_force = 13
    df['ForceIndex'] = (df['close'] - df['close'].shift(1)) * df['volumefrom']
    df['ForceIndex'] = df['ForceIndex'].rolling(period_force).mean()
    df['EMA_5']=df['EMA_5'].pct_change()
    df['EMA_30']=df['EMA_30'].pct_change()
    df['EMA_100']=df['EMA_100'].pct_change()
    #Кастомные индикаторы свечные+ другие
    df['hammer_rsi_low'] = ((df['CDL_HAMMER'] > 0) & (df['rsi'] < 30)).astype(int)
    df['engulfing_near_bb_lower'] = (
        (df['CDL_ENGULFING'] > 0) &
        (df['bb_pos'] < 0.2)
    ).astype(int)
    df['mfi_oversold_breakout'] = (
        (df['MFI'] < 20) & 
        (df['bb_break_upper'] == 1)
    ).astype(int)
    df['macd_bull_cross'] = (
        (df['macd_diff'] > 0) & 
        (df['macd_diff'].shift(1) < 0) &
        (df['rsi'] > 50)
    ).astype(int)
    df['bb_rsi_overbought'] = (
        (df['bb_break_upper'] == 1) & 
        (df['rsi'] > 70)
    ).astype(int)
    df['force_volume_spike'] = (
        (df['ForceIndex'] > 0) &
        (df['volumefrom'] > df['volume_avg'])
    ).astype(int)
    df['3crows_rsi_high'] = (
        (df['CDL_3BLACKCROWS'] < 0) & 
        (df['rsi'] > 60)
    ).astype(int)
    df['bullish_signals'] = (
        df['hammer_rsi_low'] +
        df['engulfing_near_bb_lower'] +
        df['macd_bull_cross'] +
        df['mfi_oversold_breakout']
    )

    df['bearish_signals'] = (
        df['bb_rsi_overbought'] +
        df['3crows_rsi_high']
    )

    df['doji_rsi_extreme'] = (
        (df['CDL_DOJI'] != 0) &
        ((df['rsi'] < 30) | (df['rsi'] > 70))
    ).astype(int) #Doji в зоне перекупленности / перепроданности

    df['hammer_bb_low'] = (
        (df['CDL_HAMMER'] > 0) &
        (df['bb_pos'] < 0.2)
    ).astype(int) #Молот + BB ниже 20% (глубоко внизу канала)

    df['hangingman_bearish'] = (
        (df['CDL_HANGINGMAN'] < 0) &
        (df['rsi'] > 60) &
        (df['macd_diff'] < 0)
    ).astype(int) #Повешенный + RSI > 60 + MACD убывает

    df['engulfing_with_divergence'] = (
        (df['CDL_ENGULFING'] != 0) &
        (df['divergence_strength'].abs() > df['divergence_strength'].std())
    ).astype(int) #Engulfing + высокая сила дивергенции

    df['morningstar_reversal'] = (
        (df['CDL_MORNINGSTAR'] > 0) &
        (df['bb_break_lower'] == 1)
    ).astype(int)

    df['eveningstar_reversal'] = (
        (df['CDL_EVENINGSTAR'] < 0) &
        (df['bb_break_upper'] == 1)
    ).astype(int) #Утренняя/вечерняя звезда + выход за Bollinger Band

    df['soldiers_after_acc'] = (
        (df['CDL_3WHITESOLDIERS'] > 0) &
        (df['bb_break_lower_cumulative'] > 2)
    ).astype(int) #Три белых солдата после накопления

    pattern_cols = [col for col in df.columns if col.startswith('CDL_')]
    df['bullish_patterns'] = df[pattern_cols].apply(lambda row: sum(row > 0), axis=1)
    df['bearish_patterns'] = df[pattern_cols].apply(lambda row: sum(row < 0), axis=1)
    #Количество активных бычьих и медвежьих паттернов

    #бинарный фичи с патернами
   
    
    #паттерны подряд для уверенности
    # # Шаг 1. Группа подряд идущих одинаковых значений
    # df['pattern_group'] = (
    #     (df['predicted_label'] != df['predicted_label'].shift(1)).cumsum()
    # )

    # # Шаг 2. Считаем streak — сколько раз подряд паттерн повторяется
    # df['pattern_streak'] = df.groupby('pattern_group').cumcount() + 1

    # # Шаг 3. Бинарная фича: streak >= 2 и фигура не NaN
    # df['pattern_streak_2plus'] = (
    #     (df['pattern_streak'] >= 2) &
    #     (df['predicted_label'].notnull())
    # ).astype(int)

    # # Шаг 4. Генерация бинарных фич по каждому паттерну
    # pattern_labels = [
    #     'inverse_head_and_shoulders', 'double_top',
    #     'head_and_shoulders', 'falling_wedge', 'rising_wedge',
    #     'double_bottom', 'triangle'
    # ]

    # for pattern in pattern_labels:
    #     col_name = f'{pattern}_repeated'
    #     df[col_name] = (
    #         (df['predicted_label'] == pattern) &
    #         (df['pattern_streak'] >= 2)
    #     ).astype(int)

    # df.drop(['pattern_group', 'pattern_streak'], axis=1, inplace=True)


    #Кастомные фичи с патернами

    #Сигналы после появления паттерна
    


    df["close_next_hour"] = df["close"].shift(-1)
    df["pct_change"] = df["close_next_hour"].pct_change()
    # df["target_5class"] = pd.qcut(
    #     df["pct_change"],
    #     q=5,
    #     labels=[0,1,2,3,4]
    # )
    df.reset_index(drop=True, inplace=True)
    df=df.drop(columns=['high', 'low', 'open', 'volumefrom', 'volumeto', 'close',
       'vwap',"close_next_hour",'conversionType','conversionSymbol','pct_change'])
    
    df=df[df['time']>'2020-08-01']
    df=df.fillna(0)
    return df
solana_prices_days=pd.read_csv(r"C:\Users\User\my\py\practise\solana\data\sol_price_from_2020.csv")
features_t=generate_technical(solana_prices_days)


features_t.to_csv(r'C:\Users\User\my\py\practise\solana\data\technical_2020.csv',index=False)
# этот столбец "close" больше не нужен, можно убрать


In [21]:
features_t

,time,CDL_ENGULFING,CDL_HAMMER,CDL_HANGINGMAN,CDL_DOJI,CDL_MORNINGSTAR,CDL_EVENINGSTAR,CDL_3WHITESOLDIERS,CDL_3BLACKCROWS,CDL_PIERCING,...,bearish_signals,doji_rsi_extreme,hammer_bb_low,hangingman_bearish,engulfing_with_divergence,morningstar_reversal,eveningstar_reversal,soldiers_after_acc,bullish_patterns,bearish_patterns
1980,2020-08-02,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1981,2020-08-03,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1982,2020-08-04,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1983,2020-08-05,-100,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,1
1984,2020-08-06,80,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3997,2026-02-09,0,0,0,100,0,0,0,0,0,...,0,1,0,0,0,0,0,0,1,0
3998,2026-02-10,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3999,2026-02-11,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4000,2026-02-12,0,0,0,100,0,0,0,0,0,...,0,1,0,0,0,0,0,0,1,0
